In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
batch_size = 64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081))
])

train_dataset = datasets.MNIST(root='/data',train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='/data',train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, shuffle=True, batch_size = batch_size)
test_loader = DataLoader(test_dataset, shuffle=True, batch_size = batch_size)

In [3]:
class CNN(nn.Module):

  def __init__(self):
    super(CNN,self).__init__()
    self.conv1 = nn.Conv2d(1,32,3,1)
    self.conv2 = nn.Conv2d(32,64,3,1)
    self.pool = nn.MaxPool2d(2)
    self.fc1 = nn.Linear(64*12*12,128)
    self.fc2 = nn.Linear(128,10)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(0.25)

  def forward(self, x):
    x = self.relu(self.conv1(x))
    x = self.relu(self.conv2(x))
    x = self.pool(x)
    x = torch.flatten(x,1)
    x = self.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.fc2(x)
    return x

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
epochs = 3

for epoch in range(epochs):
  total = 0
  correct = 0
  running_loss = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()

    output = model(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _, predicted = torch.max(output,1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  epoch_loss = running_loss / len(train_loader)
  accuracy = correct / total * 100

  print(f"Epoch {epoch+1}", f"Loss:{epoch_loss:.4f}", f"Accuracy:{accuracy:.2f}%")

Epoch 1 Loss:0.1370 Accuracy:95.91%
Epoch 2 Loss:0.0498 Accuracy:98.47%
Epoch 3 Loss:0.0331 Accuracy:98.97%


In [7]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for images,labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    _,predicted = torch.max(output,1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

test_acc = correct/total * 100

print(f"Test Accuracy : {test_acc:.2f}%")

Test Accuracy : 98.94%
